In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.svm import SVR
import numpy as np

# **SVM models**

In [2]:
# Load dataset
df_svm = pd.read_excel('E:\IUT\Lessons\Project-Bachelor\Husbandry\Dataset\TCI_sas (1).xlsx', sheet_name="Sheet1", usecols=["milkperiod", "zdate", "zdate_month", "firstmilk", "firstmilkdays",
                                                                                "prelendays", "drylendays", "milkdays", "previous_Milk305",
                                                                                "firstmilk_previous", "SCS_305"])

In [2]:
# Dictionary to store all model results
model_results = {}

In [ ]:
# Handle missing values
df_svm['drylendays'] = df_svm['drylendays'].fillna(df_svm['drylendays'].median())
df_svm['milkdays'] = df_svm['milkdays'].fillna(df_svm['milkdays'].median())

In [4]:
df_svm.isnull().sum()

milkperiod            0
zdate                 0
zdate_month           0
firstmilk             0
firstmilkdays         0
prelendays            0
drylendays            0
milkdays              0
previous_Milk305      0
firstmilk_previous    0
SCS_305               0
dtype: int64

In [5]:
# Separate features and target
X = df_svm.drop(columns=["firstmilk"])
y = df_svm["firstmilk"].astype(float)

In [6]:
# Split the data into train, validation, and test sets
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

In [7]:
numerical_columns = ["milkperiod","zdate","zdate_month", "firstmilkdays", "prelendays", "drylendays", "milkdays",
                     "previous_Milk305", "firstmilk_previous", "SCS_305"]

# Scale the features (SVM is sensitive to feature scaling)
scaler = StandardScaler()
scaler.fit(X_train[numerical_columns])
X_train[numerical_columns] = scaler.transform(X_train[numerical_columns])
X_val[numerical_columns] = scaler.transform(X_val[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

### **Define functions for additional metrics**


In [10]:
def calculate_mpe(y_true, y_pred):
    # Avoid division by zero
    mask = y_true != 0
    return np.mean((y_true[mask] - y_pred[mask]) / y_true[mask] * 100)

def calculate_smape(y_true, y_pred):
    # Avoid division by zero
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask] * 100)

def calculate_sdr(y_true, y_pred):
    return np.std(y_pred) / np.std(y_true)

# **Linear Kernel**

In [11]:
# **Linear Kernel**
print("\n=== Block 1: Linear Kernel ===")
kernel = 'linear'
svr_linear = SVR(kernel=kernel)
svr_linear.fit(X_train[numerical_columns], y_train)

# Predict on test set
y_test_pred_linear = svr_linear.predict(X_test[numerical_columns])

# Calculate metrics
test_r2_linear = r2_score(y_test, y_test_pred_linear)
test_mae_linear = mean_absolute_error(y_test, y_test_pred_linear)
test_mse_linear = mean_squared_error(y_test, y_test_pred_linear)
test_rmse_linear = np.sqrt(test_mse_linear)
test_mpe_linear = calculate_mpe(y_test, y_test_pred_linear)
test_smape_linear = calculate_smape(y_test, y_test_pred_linear)
test_sdr_linear = calculate_sdr(y_test, y_test_pred_linear)

# Print results
print("Test Set (Linear):")
print(f"R²    : {test_r2_linear:.4f}")
print(f"MAE   : {test_mae_linear:.4f}")
print(f"RMSE  : {test_rmse_linear:.4f}")
print(f"MPE   : {test_mpe_linear:.4f}")
print(f"sMAPE : {test_smape_linear:.4f}")
print(f"SDR   : {test_sdr_linear:.4f}")


=== Block 1: Linear Kernel ===
Test Set (Linear):
R²    : 0.3071
MAE   : 6.9707
RMSE  : 9.2327
MPE   : -9.7312
sMAPE : 17.5841
SDR   : 0.5847


In [ ]:
# Dictionary to store results for each kernel
results = {}

# Block 1: Linear Kernel
print("\n=== Block 1: Linear Kernel ===")
kernel = 'linear'
svr_linear = SVR(kernel=kernel)
svr_linear.fit(X_train[numerical_columns], y_train)

# Predict on all sets
y_train_pred_linear = svr_linear.predict(X_train[numerical_columns])
y_val_pred_linear = svr_linear.predict(X_val[numerical_columns])
y_test_pred_linear = svr_linear.predict(X_test[numerical_columns])

# Calculate metrics
train_r2_linear = r2_score(y_train, y_train_pred_linear)
val_r2_linear = r2_score(y_val, y_val_pred_linear)
test_r2_linear = r2_score(y_test, y_test_pred_linear)
train_mae_linear = mean_absolute_error(y_train, y_train_pred_linear)
val_mae_linear = mean_absolute_error(y_val, y_val_pred_linear)
test_mae_linear = mean_absolute_error(y_test, y_test_pred_linear)
train_mse_linear = mean_squared_error(y_train, y_train_pred_linear)
val_mse_linear = mean_squared_error(y_val, y_val_pred_linear)
test_mse_linear = mean_squared_error(y_test, y_test_pred_linear)

# Print results
print("Train Set (Linear):")
print(f"R²   : {train_r2_linear:.4f}")
print(f"MAE  : {train_mae_linear:.4f}")
print(f"MSE  : {train_mse_linear:.4f}")
print("\nValidation Set (Linear):")
print(f"R²   : {val_r2_linear:.4f}")
print(f"MAE  : {val_mae_linear:.4f}")
print(f"MSE  : {val_mse_linear:.4f}")
print("\nTest Set (Linear):")
print(f"R²   : {test_r2_linear:.4f}")
print(f"MAE  : {test_mae_linear:.4f}")
print(f"MSE  : {test_mse_linear:.4f}")

# Overfitting Check
r2_gap_linear = train_r2_linear - val_r2_linear
r2_gap_threshold = 0.05
print("\nOverfitting Analysis (Linear):")
print("=" * 50)
if r2_gap_linear > r2_gap_threshold:
    print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap_linear:.4f}) is larger than threshold ({r2_gap_threshold}).")
else:
    print(f"No significant overfitting based on Train-Val R² Gap ({r2_gap_linear:.4f}).")

results['linear'] = {
    'train_r2': train_r2_linear,
    'val_r2': val_r2_linear,
    'test_r2': test_r2_linear,
    'train_mse': train_mse_linear,
    'val_mse': val_mse_linear,
    'test_mse': test_mse_linear,
    'model': svr_linear
}


=== Block 1: Linear Kernel ===
Train Set (Linear):
R²   : 0.3075
MAE  : 6.9205
MSE  : 83.9880

Validation Set (Linear):
R²   : 0.3001
MAE  : 6.9461
MSE  : 85.1907

Test Set (Linear):
R²   : 0.3071
MAE  : 6.9707
MSE  : 85.2431

Overfitting Analysis (Linear):
No significant overfitting based on Train-Val R² Gap (0.0073).


In [ ]:
 # Store results for SVR
model_results[f'SVR_Linear'] = {
    'Train_R2': train_r2_linear, 'Train_MAE': train_mae_linear, 'Train_MSE': train_mse_linear,
    'Val_R2': val_r2_linear, 'Val_MAE': val_mae_linear, 'Val_MSE': val_mse_linear,
    'Test_R2': test_r2_linear, 'Test_MAE': test_mae_linear, 'Test_MSE': test_mse_linear
}

## **Polynomial Kernel**


In [12]:
# **Polynomial Kernel**
print("\n=== Block 2: Polynomial Kernel ===")
kernel = 'poly'
svr_poly = SVR(kernel=kernel)
svr_poly.fit(X_train[numerical_columns], y_train)

# Predict on test set
y_test_pred_poly = svr_poly.predict(X_test[numerical_columns])

# Calculate metrics
test_r2_poly = r2_score(y_test, y_test_pred_poly)
test_mae_poly = mean_absolute_error(y_test, y_test_pred_poly)
test_mse_poly = mean_squared_error(y_test, y_test_pred_poly)
test_rmse_poly = np.sqrt(test_mse_poly)
test_mpe_poly = calculate_mpe(y_test, y_test_pred_poly)
test_smape_poly = calculate_smape(y_test, y_test_pred_poly)
test_sdr_poly = calculate_sdr(y_test, y_test_pred_poly)

# Print results
print("Test Set (Poly):")
print(f"R²    : {test_r2_poly:.4f}")
print(f"MAE   : {test_mae_poly:.4f}")
print(f"RMSE  : {test_rmse_poly:.4f}")
print(f"MPE   : {test_mpe_poly:.4f}")
print(f"sMAPE : {test_smape_poly:.4f}")
print(f"SDR   : {test_sdr_poly:.4f}")


=== Block 2: Polynomial Kernel ===
Test Set (Poly):
R²    : 0.2648
MAE   : 7.2162
RMSE  : 9.5106
MPE   : -9.8205
sMAPE : 18.1761
SDR   : 0.5548


In [ ]:
# Block 2: Polynomial Kernel
print("\n=== Block 2: Polynomial Kernel ===")
kernel = 'poly'
svr_poly = SVR(kernel=kernel)
svr_poly.fit(X_train[numerical_columns], y_train)

# Predict on all sets
y_train_pred_poly = svr_poly.predict(X_train[numerical_columns])
y_val_pred_poly = svr_poly.predict(X_val[numerical_columns])
y_test_pred_poly = svr_poly.predict(X_test[numerical_columns])

# Calculate metrics
train_r2_poly = r2_score(y_train, y_train_pred_poly)
val_r2_poly = r2_score(y_val, y_val_pred_poly)
test_r2_poly = r2_score(y_test, y_test_pred_poly)
train_mae_poly = mean_absolute_error(y_train, y_train_pred_poly)
val_mae_poly = mean_absolute_error(y_val, y_val_pred_poly)
test_mae_poly = mean_absolute_error(y_test, y_test_pred_poly)
train_mse_poly = mean_squared_error(y_train, y_train_pred_poly)
val_mse_poly = mean_squared_error(y_val, y_val_pred_poly)
test_mse_poly = mean_squared_error(y_test, y_test_pred_poly)

# Print results
print("Train Set (Poly):")
print(f"R²   : {train_r2_poly:.4f}")
print(f"MAE  : {train_mae_poly:.4f}")
print(f"MSE  : {train_mse_poly:.4f}")
print("\nValidation Set (Poly):")
print(f"R²   : {val_r2_poly:.4f}")
print(f"MAE  : {val_mae_poly:.4f}")
print(f"MSE  : {val_mse_poly:.4f}")
print("\nTest Set (Poly):")
print(f"R²   : {test_r2_poly:.4f}")
print(f"MAE  : {test_mae_poly:.4f}")
print(f"MSE  : {test_mse_poly:.4f}")

# Overfitting Check
r2_gap_poly = train_r2_poly - val_r2_poly
print("\nOverfitting Analysis (Poly):")
print("=" * 50)
if r2_gap_poly > r2_gap_threshold:
    print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap_poly:.4f}) is larger than threshold ({r2_gap_threshold}).")
else:
    print(f"No significant overfitting based on Train-Val R² Gap ({r2_gap_poly:.4f}).")

results['poly'] = {
    'train_r2': train_r2_poly,
    'val_r2': val_r2_poly,
    'test_r2': test_r2_poly,
    'train_mse': train_mse_poly,
    'val_mse': val_mse_poly,
    'test_mse': test_mse_poly,
    'model': svr_poly
}


=== Block 2: Polynomial Kernel ===
Train Set (Poly):
R²   : 0.2679
MAE  : 7.1516
MSE  : 88.7907

Validation Set (Poly):
R²   : 0.2478
MAE  : 7.2135
MSE  : 91.5550

Test Set (Poly):
R²   : 0.2648
MAE  : 7.2162
MSE  : 90.4523

Overfitting Analysis (Poly):
No significant overfitting based on Train-Val R² Gap (0.0200).


In [ ]:
 # Store results for SVR
model_results[f'SVR_Polynomial'] = {
    'Train_R2': train_r2_poly, 'Train_MAE': train_mae_poly, 'Train_MSE': train_mse_poly,
    'Val_R2': val_r2_poly, 'Val_MAE': val_mae_poly, 'Val_MSE': val_mse_poly,
    'Test_R2': test_r2_poly, 'Test_MAE': test_mae_poly, 'Test_MSE': test_mse_poly
}

## **RBF Kernel**


In [13]:
# **RBF Kernel**
print("\n=== Block 3: RBF Kernel ===")
kernel = 'rbf'
svr_rbf = SVR(kernel=kernel)
svr_rbf.fit(X_train[numerical_columns], y_train)

# Predict on test set
y_test_pred_rbf = svr_rbf.predict(X_test[numerical_columns])

# Calculate metrics
test_r2_rbf = r2_score(y_test, y_test_pred_rbf)
test_mae_rbf = mean_absolute_error(y_test, y_test_pred_rbf)
test_mse_rbf = mean_squared_error(y_test, y_test_pred_rbf)
test_rmse_rbf = np.sqrt(test_mse_rbf)
test_mpe_rbf = calculate_mpe(y_test, y_test_pred_rbf)
test_smape_rbf = calculate_smape(y_test, y_test_pred_rbf)
test_sdr_rbf = calculate_sdr(y_test, y_test_pred_rbf)

# Print results
print("Test Set (RBF):")
print(f"R²    : {test_r2_rbf:.4f}")
print(f"MAE   : {test_mae_rbf:.4f}")
print(f"RMSE  : {test_rmse_rbf:.4f}")
print(f"MPE   : {test_mpe_rbf:.4f}")
print(f"sMAPE : {test_smape_rbf:.4f}")
print(f"SDR   : {test_sdr_rbf:.4f}")


=== Block 3: RBF Kernel ===
Test Set (RBF):
R²    : 0.3643
MAE   : 6.6377
RMSE  : 8.8437
MPE   : -9.2378
sMAPE : 16.7586
SDR   : 0.6156


In [ ]:
# Block 3: RBF Kernel
print("\n=== Block 3: RBF Kernel ===")
kernel = 'rbf'
svr_rbf = SVR(kernel=kernel)
svr_rbf.fit(X_train[numerical_columns], y_train)

# Predict on all sets
y_train_pred_rbf = svr_rbf.predict(X_train[numerical_columns])
y_val_pred_rbf = svr_rbf.predict(X_val[numerical_columns])
y_test_pred_rbf = svr_rbf.predict(X_test[numerical_columns])

# Calculate metrics
train_r2_rbf = r2_score(y_train, y_train_pred_rbf)
val_r2_rbf = r2_score(y_val, y_val_pred_rbf)
test_r2_rbf = r2_score(y_test, y_test_pred_rbf)
train_mae_rbf = mean_absolute_error(y_train, y_train_pred_rbf)
val_mae_rbf = mean_absolute_error(y_val, y_val_pred_rbf)
test_mae_rbf = mean_absolute_error(y_test, y_test_pred_rbf)
train_mse_rbf = mean_squared_error(y_train, y_train_pred_rbf)
val_mse_rbf = mean_squared_error(y_val, y_val_pred_rbf)
test_mse_rbf = mean_squared_error(y_test, y_test_pred_rbf)

# Print results
print("Train Set (RBF):")
print(f"R²   : {train_r2_rbf:.4f}")
print(f"MAE  : {train_mae_rbf:.4f}")
print(f"MSE  : {train_mse_rbf:.4f}")
print("\nValidation Set (RBF):")
print(f"R²   : {val_r2_rbf:.4f}")
print(f"MAE  : {val_mae_rbf:.4f}")
print(f"MSE  : {val_mse_rbf:.4f}")
print("\nTest Set (RBF):")
print(f"R²   : {test_r2_rbf:.4f}")
print(f"MAE  : {test_mae_rbf:.4f}")
print(f"MSE  : {test_mse_rbf:.4f}")

# Overfitting Check
r2_gap_rbf = train_r2_rbf - val_r2_rbf
print("\nOverfitting Analysis (RBF):")
print("=" * 50)
if r2_gap_rbf > r2_gap_threshold:
    print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap_rbf:.4f}) is larger than threshold ({r2_gap_threshold}).")
else:
    print(f"No significant overfitting based on Train-Val R² Gap ({r2_gap_rbf:.4f}).")

results['rbf'] = {
    'train_r2': train_r2_rbf,
    'val_r2': val_r2_rbf,
    'test_r2': test_r2_rbf,
    'train_mse': train_mse_rbf,
    'val_mse': val_mse_rbf,
    'test_mse': test_mse_rbf,
    'model': svr_rbf
}


=== Block 3: RBF Kernel ===
Train Set (RBF):
R²   : 0.3696
MAE  : 6.5594
MSE  : 76.4491

Validation Set (RBF):
R²   : 0.3559
MAE  : 6.6434
MSE  : 78.4056

Test Set (RBF):
R²   : 0.3643
MAE  : 6.6377
MSE  : 78.2113

Overfitting Analysis (RBF):
No significant overfitting based on Train-Val R² Gap (0.0138).


In [ ]:
 # Store results for SVR
model_results[f'SVR_RBF'] = {
    'Train_R2': train_r2_rbf, 'Train_MAE': train_mae_rbf, 'Train_MSE': train_mse_rbf,
    'Val_R2': val_r2_rbf, 'Val_MAE': val_mae_rbf, 'Val_MSE': val_mse_rbf,
    'Test_R2': test_r2_rbf, 'Test_MAE': test_mae_rbf, 'Test_MSE': test_mse_rbf
}

# **RBF SVR with robust scaler and drop miss value**

In [14]:
from sklearn.preprocessing import StandardScaler, RobustScaler
# **RBF SVR with RobustScaler and drop missing values**
# Load dataset
df_svm_robust = pd.read_excel('E:\IUT\Lessons\Project-Bachelor\Husbandry\Dataset\TCI_sas (1).xlsx', sheet_name="Sheet1", usecols=["milkperiod", "zdate", "zdate_month", "firstmilk", "firstmilkdays",
                                                                                 "prelendays", "drylendays", "milkdays", "previous_Milk305",
                                                                                 "firstmilk_previous", "SCS_305"])

# Handle missing values by dropping NaN
df_svm_robust = df_svm_robust.dropna()

# Separate features and target
X_robust = df_svm_robust.drop(columns=["firstmilk"])
y_robust = df_svm_robust["firstmilk"].astype(float)

# Split the data into train, validation, and test sets
X_temp_robust, X_test_robust, y_temp_robust, y_test_robust = train_test_split(X_robust, y_robust, test_size=0.15, random_state=42)
X_train_robust, X_val_robust, y_train_robust, y_val_robust = train_test_split(X_temp_robust, y_temp_robust, test_size=0.1765, random_state=42)

numerical_columns_robust = ["milkperiod", "zdate", "zdate_month", "firstmilkdays", "prelendays", "drylendays", "milkdays", "previous_Milk305", "firstmilk_previous", "SCS_305"]

# Scale the features using RobustScaler
scaler_robust = RobustScaler()
X_train_robust_scaled = scaler_robust.fit_transform(X_train_robust[numerical_columns_robust])
X_test_robust_scaled = scaler_robust.transform(X_test_robust[numerical_columns_robust])

print("\n=== Block 4: RBF Kernel with RobustScaler ===")
kernel = 'rbf'
svr_rbf_robust = SVR(kernel=kernel, verbose=3)  # Use verbose to see training progress
svr_rbf_robust.fit(X_train_robust_scaled, y_train_robust)

# Predict on test set
y_test_pred_rbf_robust = svr_rbf_robust.predict(X_test_robust_scaled)

# Calculate metrics
test_r2_rbf_robust = r2_score(y_test_robust, y_test_pred_rbf_robust)
test_mae_rbf_robust = mean_absolute_error(y_test_robust, y_test_pred_rbf_robust)
test_mse_rbf_robust = mean_squared_error(y_test_robust, y_test_pred_rbf_robust)
test_rmse_rbf_robust = np.sqrt(test_mse_rbf_robust)
test_mpe_rbf_robust = calculate_mpe(y_test_robust, y_test_pred_rbf_robust)
test_smape_rbf_robust = calculate_smape(y_test_robust, y_test_pred_rbf_robust)
test_sdr_rbf_robust = calculate_sdr(y_test_robust, y_test_pred_rbf_robust)

# Print results
print("Test Set (RBF with RobustScaler):")
print(f"R²    : {test_r2_rbf_robust:.4f}")
print(f"MAE   : {test_mae_rbf_robust:.4f}")
print(f"RMSE  : {test_rmse_rbf_robust:.4f}")
print(f"MPE   : {test_mpe_rbf_robust:.4f}")
print(f"sMAPE : {test_smape_rbf_robust:.4f}")
print(f"SDR   : {test_sdr_rbf_robust:.4f}")


=== Block 4: RBF Kernel with RobustScaler ===
[LibSVM]Test Set (RBF with RobustScaler):
R²    : 0.3601
MAE   : 6.6438
RMSE  : 8.8610
MPE   : -9.2403
sMAPE : 16.7757
SDR   : 0.6166


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
df_svm = pd.read_excel('../Dataset/TCI_sas (1).xlsx', sheet_name="Sheet1", usecols=["milkperiod", "zdate", "zdate_month", "firstmilk", "firstmilkdays",
                                                                                "prelendays", "drylendays", "milkdays", "previous_Milk305",
                                                                                "firstmilk_previous", "SCS_305"])

# Handle missing values by dropping NaN
df_svm = df_svm.dropna()

# Separate features and target
X = df_svm.drop(columns=["firstmilk"])
y = df_svm["firstmilk"].astype(float)

# Split the data into train, validation, and test sets
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

numerical_columns = ["milkperiod", "zdate", "zdate_month", "firstmilkdays", "prelendays", "drylendays", "milkdays", "previous_Milk305", "firstmilk_previous", "SCS_305"]

# Scale the features using RobustScaler
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train[numerical_columns])
X_val_scaled = scaler.transform(X_val[numerical_columns])
X_test_scaled = scaler.transform(X_test[numerical_columns])

In [ ]:
# Block 3: RBF Kernel
print("\n=== Block 3: RBF Kernel ===")
kernel = 'rbf'
svr_rbf = SVR(kernel=kernel, verbose=3)  # Use verbose to see training progress
svr_rbf.fit(X_train_scaled, y_train)

# Predict on all sets
y_train_pred_rbf = svr_rbf.predict(X_train_scaled)
y_val_pred_rbf = svr_rbf.predict(X_val_scaled)
y_test_pred_rbf = svr_rbf.predict(X_test_scaled)

# Calculate metrics
train_r2_rbf = r2_score(y_train, y_train_pred_rbf)
val_r2_rbf = r2_score(y_val, y_val_pred_rbf)
test_r2_rbf = r2_score(y_test, y_test_pred_rbf)
train_mae_rbf = mean_absolute_error(y_train, y_train_pred_rbf)
val_mae_rbf = mean_absolute_error(y_val, y_val_pred_rbf)
test_mae_rbf = mean_absolute_error(y_test, y_test_pred_rbf)
train_mse_rbf = mean_squared_error(y_train, y_train_pred_rbf)
val_mse_rbf = mean_squared_error(y_val, y_val_pred_rbf)
test_mse_rbf = mean_squared_error(y_test, y_test_pred_rbf)

# Print results
print("Train Set (RBF):")
print(f"R²   : {train_r2_rbf:.4f}")
print(f"MAE  : {train_mae_rbf:.4f}")
print(f"MSE  : {train_mse_rbf:.4f}")
print("\nValidation Set (RBF):")
print(f"R²   : {val_r2_rbf:.4f}")
print(f"MAE  : {val_mae_rbf:.4f}")
print(f"MSE  : {val_mse_rbf:.4f}")
print("\nTest Set (RBF):")
print(f"R²   : {test_r2_rbf:.4f}")
print(f"MAE  : {test_mae_rbf:.4f}")
print(f"MSE  : {test_mse_rbf:.4f}")

# Overfitting Check
r2_gap_threshold = 0.05
r2_gap_rbf = train_r2_rbf - val_r2_rbf
print("\nOverfitting Analysis (RBF):")
print("=" * 50)
if r2_gap_rbf > r2_gap_threshold:
    print(f"Warning: Potential Overfitting Detected! Train-Val R² Gap ({r2_gap_rbf:.4f}) is larger than threshold ({r2_gap_threshold}).")
else:
    print(f"No significant overfitting based on Train-Val R² Gap ({r2_gap_rbf:.4f}).")



=== Block 3: RBF Kernel ===
[LibSVM]Train Set (RBF):
R²   : 0.3673
MAE  : 6.5863
MSE  : 76.8592

Validation Set (RBF):
R²   : 0.3609
MAE  : 6.6058
MSE  : 77.2465

Test Set (RBF):
R²   : 0.3601
MAE  : 6.6438
MSE  : 78.5178

Overfitting Analysis (RBF):
No significant overfitting based on Train-Val R² Gap (0.0063).


In [ ]:
# Function to print existing CSV content
def print_csv_content(file_path):
    try:
        existing_df = pd.read_csv(file_path, index_col=0)
        return existing_df
    except FileNotFoundError:
        print("\nNo existing comparison table found.")
        return pd.DataFrame()

# Load existing CSV content from parent directory
file_path = '../model_comparison_table (1) (1).csv'
existing_df = print_csv_content(file_path)

# Save results to CSV (new results appended)
results = {}  # Initialize results dictionary
results['rbf_robust'] = {
    'train_r2': train_r2_rbf,
    'val_r2': val_r2_rbf,
    'test_r2': test_r2_rbf,
    'train_mse': train_mse_rbf,
    'val_mse': val_mse_rbf,
    'test_mse': test_mse_rbf,
    'model': svr_rbf
}

# Convert results to DataFrame
new_results_df = pd.DataFrame.from_dict({
    'SVR_RBF_Robust': {
        'Train_R2': train_r2_rbf, 'Train_MAE': train_mae_rbf, 'Train_MSE': train_mse_rbf,
        'Val_R2': val_r2_rbf, 'Val_MAE': val_mae_rbf, 'Val_MSE': val_mse_rbf,
        'Test_R2': test_r2_rbf, 'Test_MAE': test_mae_rbf, 'Test_MSE': test_mse_rbf
    }
}, orient='index')

# If existing data exists, append it; otherwise, start with new results
if not existing_df.empty:
    comparison_df = pd.concat([existing_df, new_results_df])
else:
    comparison_df = new_results_df

# Save to the parent directory file
comparison_df.to_csv(file_path, index=True)
print("\n=== Updated Model Comparison Table ===")
print(comparison_df)


=== Updated Model Comparison Table ===
                                                    Train_R2  Train_MAE  \
Ridge_Initial                                       0.311672   6.977889   
Ridge_Optimized                                     0.313746   6.956954   
RF_Initial                                          0.908400   2.525600   
RF_Tuned                                            0.379600   6.612000   
RF_CV_Final                                              NaN        NaN   
RF_Initial_with_grp                                 0.907400   2.541200   
RF_with_grp_Tuned                                   0.378200   6.620800   
RF_with_grp_CV_Final                                     NaN        NaN   
SVR_linear                                          0.307500   6.920500   
SVR_poly                                            0.267900   7.151600   
SVR_rbf                                             0.369600   6.559400   
XGBoost                                             0.416776

In [15]:
## **Print results into a new CSV file**
# Function to print existing CSV content
def print_csv_content(file_path):
    try:
        existing_df = pd.read_csv(file_path, index_col=0)
        return existing_df
    except FileNotFoundError:
        print("\nNo existing test evaluation table found.")
        return pd.DataFrame()

# Load existing CSV content from new file (if exists)
new_file_path = 'test_evaluation_metrics.csv'
existing_df = print_csv_content(new_file_path)

# Save results to a new CSV file
linear_results = {
    'SVR_Linear': {
        'Test_R2': test_r2_linear, 'Test_MAE': test_mae_linear, 'Test_RMSE': test_rmse_linear,
        'Test_MPE': test_mpe_linear, 'Test_sMAPE': test_smape_linear, 'Test_SDR': test_sdr_linear
    }
}
linear_df = pd.DataFrame.from_dict(linear_results, orient='index')

poly_results = {
    'SVR_Polynomial': {
        'Test_R2': test_r2_poly, 'Test_MAE': test_mae_poly, 'Test_RMSE': test_rmse_poly,
        'Test_MPE': test_mpe_poly, 'Test_sMAPE': test_smape_poly, 'Test_SDR': test_sdr_poly
    }
}
poly_df = pd.DataFrame.from_dict(poly_results, orient='index')

rbf_results = {
    'SVR_RBF': {
        'Test_R2': test_r2_rbf, 'Test_MAE': test_mae_rbf, 'Test_RMSE': test_rmse_rbf,
        'Test_MPE': test_mpe_rbf, 'Test_sMAPE': test_smape_rbf, 'Test_SDR': test_sdr_rbf
    }
}
rbf_df = pd.DataFrame.from_dict(rbf_results, orient='index')

rbf_robust_results = {
    'SVR_RBF_Robust': {
        'Test_R2': test_r2_rbf_robust, 'Test_MAE': test_mae_rbf_robust, 'Test_RMSE': test_rmse_rbf_robust,
        'Test_MPE': test_mpe_rbf_robust, 'Test_sMAPE': test_smape_rbf_robust, 'Test_SDR': test_sdr_rbf_robust
    }
}
rbf_robust_df = pd.DataFrame.from_dict(rbf_robust_results, orient='index')

# Combine new results
new_results_df = pd.concat([linear_df, poly_df, rbf_df, rbf_robust_df])

# If existing data exists, append it; otherwise, start with new results
if not existing_df.empty:
    comparison_df = pd.concat([new_results_df, existing_df])
else:
    comparison_df = new_results_df

# Save to the new file
comparison_df.to_csv(new_file_path, index=True)
print("\n=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===")
print(comparison_df)


=== Updated Test Evaluation Metrics Saved to test_evaluation_metrics.csv ===
                       Test_R2  Test_MAE  Test_RMSE  Test_MPE  Test_sMAPE  \
SVR_Linear            0.307137  6.970674   9.232719 -9.731248   17.584111   
SVR_Polynomial        0.264797  7.216206   9.510640 -9.820527   18.176112   
SVR_RBF               0.364292  6.637658   8.843716 -9.237821   16.758594   
SVR_RBF_Robust        0.360090  6.643834   8.861028 -9.240320   16.775699   
RF_Initial            0.347709  6.828523   8.958325 -6.936344   17.222969   
RF_Tuned              0.357528  6.759105   8.890640 -7.315823   17.055061   
RF_CV_Final           0.352554  6.784989   8.924994 -7.610628   17.092712   
RF_Initial_with_grp   0.340054  6.878539   9.010737 -6.964281   17.342442   
RF_with_grp           0.351373  6.792523   8.933128 -7.608877   17.112260   
RF_with_grp_CV_Final  0.351373  6.792523   8.933128 -7.608877   17.112260   
Ridge_Initial         0.310995  7.027556   9.206984 -8.085323   17.683976  